# Part 2 — Tourism Recommendation Engine**Preserving Heritage: Enhancing Tourism with AI** · AIML Capstone 2Understanding tourists and their expectations is critical for marketingIndonesian destinations. This notebook performs exploratory data analysis onvisitor ratings across five major Indonesian cities, then builds a**collaborative filtering** recommender that answers the agency's question:*given the place a tourist is at now, where should they go next?*### Datasets| File | Contents ||---|---|| `user.csv` | User_Id, Location, Age — visitor demographics || `tourism_with_id.xlsx` | Place_Id, Place_Name, Description, Category, City, Price, Rating, Time_Minutes, Coordinate, Lat, Long || `tourism_rating.csv` | User_Id, Place_Id, Place_Ratings — the interaction matrix |> The brief refers to `tourism_with_id.csv`; the supplied file is an `.xlsx`.> The loader accepts either.### How this notebook maps to the brief| Brief task | Where ||---|---|| 1. Import datasets; check missing values and duplicates; remove anomalies | §2 || 2. Explore the rating user group — age distribution, where tourists come from | §3 || 3. Categories of tourist spots; what each location is famous for; best city for a nature enthusiast | §4 || 4. Combine places with user ratings; most-loved spots and city; most-liked category | §5 || 5. Build a collaborative filtering recommender; recommend places from the current place name | §7–9 |

## 1. Setup

In [ ]:
import os, sys, subprocessfrom pathlib import Pathtry:    import google.colab  # noqa: F401    IN_COLAB = Trueexcept ImportError:    IN_COLAB = FalseREPO_URL  = "https://github.com/USERNAME/heritage-tourism-ai.git"   # <-- editREPO_NAME = "heritage-tourism-ai"if IN_COLAB:    if not Path(REPO_NAME).exists():        subprocess.run(["git", "clone", REPO_URL], check=False)    if Path(REPO_NAME).exists():        sys.path.insert(0, str(Path(REPO_NAME).resolve()))else:    sys.path.insert(0, str(Path.cwd().parent))# Expected Drive layout:#   MyDrive/heritage-data/tourism/{user.csv, tourism_rating.csv, tourism_with_id.xlsx}DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd().parent / "data"if IN_COLAB:    from google.colab import drive    drive.mount("/content/drive", force_remount=False)    src_dir = Path("/content/drive/MyDrive/heritage-data/tourism")    assert src_dir.exists(), f"Not found: {src_dir}\nUpload the three Part 2 files there."    (DATA_ROOT / "tourism").mkdir(parents=True, exist_ok=True)    subprocess.run(f'cp "{src_dir}"/* "{DATA_ROOT}/tourism/"', shell=True, check=True)os.environ["HERITAGE_DATA_ROOT"] = str(DATA_ROOT)print("DATA_ROOT:", DATA_ROOT)

In [ ]:
import importlibimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport src.config as configimportlib.reload(config)from src.config import RecoConfigfrom src.data import tourism_data as tdfrom src.evaluation import ranking as Rfrom src.models.item_cf import ItemBasedCF, PopularityBaselinefrom src.viz.plots import ACCENT, CONTRAST, PALETTE, plot_bar, use_house_styleuse_house_style()pd.set_option("display.max_columns", 40)pd.set_option("display.width", 160)cfg = RecoConfig(tourism_dir=DATA_ROOT / "tourism", top_k=10, embedding_dim=32)print("tourism dir exists:", cfg.tourism_dir.exists())

## 2. Import, inspect and clean — *brief task 1*The loader normalises column names to a canonical lower-case form and drops thetwo trailing unnamed columns this dataset is known to ship with, so nothingdownstream depends on the exact capitalisation in the source file.

In [ ]:
raw = td.load_raw(cfg)for name, df in raw.items():    print(f"{name:8} shape={df.shape}")    display(df.head(3))

In [ ]:
# Task 1.I - missing values, duplicates, dtypesfor name in ["places", "ratings", "users"]:    display(td.inspect(raw[name], name))

**What the inspection shows.** `time_minutes` is roughly half missing. Wedeliberately leave it as `NaN` rather than filling it with the column mean:inventing tour durations for 200+ attractions would put fabricated numbers intothe EDA and any figure computed from them would be meaningless. It is simplyexcluded from analyses that need it.

In [ ]:
# Task 1.II - remove duplicates and anomaliesdata = td.clean(raw, cfg)places, ratings, users = data["places"], data["ratings"], data["users"]

The rating matrix is around **92% sparse** — roughly 10,000 ratings spread over300 users and 437 places. Hold onto that number: it is the single mostimportant constraint on this project and it dictates the model sizing in §8.## 3. Who is rating? — *brief task 2*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))axes[0].hist(users["age"], bins=range(int(users["age"].min()), int(users["age"].max()) + 3),             color=ACCENT, edgecolor="white")axes[0].axvline(users["age"].mean(), color=CONTRAST, ls="--", lw=2)axes[0].text(users["age"].mean(), axes[0].get_ylim()[1] * 0.92,             f"  mean {users['age'].mean():.1f}", color=CONTRAST, fontsize=10)axes[0].set(xlabel="age", ylabel="users", title="Age distribution of rating users")bands = pd.cut(users["age"], bins=[0, 24, 29, 34, 39, 100],               labels=["<25", "25-29", "30-34", "35-39", "40+"])counts = bands.value_counts().sort_index()axes[1].bar(counts.index.astype(str), counts.values, color=ACCENT)for x, v in enumerate(counts.values):    axes[1].text(x, v, str(v), ha="center", va="bottom", fontsize=9, color="#444")axes[1].set(xlabel="age band", ylabel="users", title="Users by age band")fig.tight_layout()print(users["age"].describe().round(2).to_string())

In [ ]:
# Where are these tourists coming from?top_locations = users["location"].value_counts().head(15)fig = plot_bar(top_locations.index[::-1], top_locations.values[::-1],               title="Top 15 home locations of rating users",               ylabel="users", value_fmt="{:.0f}")

In [ ]:
if "province" in users.columns:    prov = users["province"].value_counts().head(10)    fig = plot_bar(prov.index[::-1], prov.values[::-1],                   title="Users by province", ylabel="users", value_fmt="{:.0f}")    display(prov.to_frame("users"))

**Finding.** The rating population is young — concentrated in the twenties andthirties — and heavily drawn from Java, with Bekasi, Jakarta and the surroundingJabodetabek area dominating. Two implications for the agency's marketing:1. Campaigns built on this data are calibrated to a **young domestic** audience.   Extrapolating these preferences to older or international visitors is not   supported by the data.2. Because most raters live near Jakarta, destinations in and around Java get   disproportionate rating volume. That is a sampling artefact, not evidence   that Javanese attractions are objectively better.## 4. Where and what are the tourist spots? — *brief task 3*

In [ ]:
# 3.I - what categories exist?cat_counts = places["category"].value_counts()display(cat_counts.to_frame("places"))fig = plot_bar(cat_counts.index[::-1], cat_counts.values[::-1],               title="Tourist spots by category", ylabel="places", value_fmt="{:.0f}")

In [ ]:
# 3.II - what is each city most famous for?city_cat = pd.crosstab(places["city"], places["category"])display(city_cat)share = city_cat.div(city_cat.sum(axis=1), axis=0) * 100fig, ax = plt.subplots(figsize=(10, 5))bottom = np.zeros(len(share))for colour, col in zip(PALETTE, share.columns):    ax.bar(share.index, share[col], bottom=bottom, label=col, color=colour)    bottom += share[col].to_numpy()ax.set(ylabel="% of the city's attractions", title="Category mix by city")ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")plt.setp(ax.get_xticklabels(), rotation=20, ha="right")fig.tight_layout()print("\nEach city's dominant category:")display(city_cat.idxmax(axis=1).to_frame("most common category"))

In [ ]:
# 3.III - which city is best for a nature enthusiast?NATURE = ["Cagar Alam", "Bahari"]   # nature reserve, marine/coastalpresent = [c for c in NATURE if c in places["category"].unique()]print("Nature categories found:", present)nature = places[places["category"].isin(present)]nature_stats = (    nature.groupby("city")    .agg(nature_spots=("place_id", "count"), avg_listed_rating=("rating", "mean"))    .join(places.groupby("city").size().rename("total_spots")))nature_stats["nature_share_%"] = (    100 * nature_stats["nature_spots"] / nature_stats["total_spots"]).round(1)nature_stats["avg_listed_rating"] = nature_stats["avg_listed_rating"].round(2)display(nature_stats.sort_values("nature_spots", ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))s = nature_stats.sort_values("nature_spots")axes[0].barh(s.index, s["nature_spots"], color=ACCENT)for y, v in enumerate(s["nature_spots"]):    axes[0].text(v, y, f" {int(v)}", va="center", fontsize=9, color="#444")axes[0].set(xlabel="nature attractions", title="Volume of nature spots")s2 = nature_stats.sort_values("nature_share_%")axes[1].barh(s2.index, s2["nature_share_%"], color=CONTRAST)for y, v in enumerate(s2["nature_share_%"]):    axes[1].text(v, y, f" {v:.1f}%", va="center", fontsize=9, color="#444")axes[1].set(xlabel="% of the city's attractions", title="Concentration of nature spots")fig.tight_layout()

**Answering 3.III properly.** "Best city for a nature enthusiast" splits intotwo different questions, and the two charts above disagree by design:- **Most choice** — the city with the largest *number* of nature attractions,  best if the visitor wants many options in one trip.- **Most concentrated** — the city with the highest *share* of nature  attractions, best if the visitor wants a trip that is nature-focused  end-to-end rather than diluted by malls and theme parks.State which definition you are using when you quote a single answer. The tableabove gives both, plus the average listed rating so quality is not ignored.## 5. Places joined to ratings — *brief task 4*

In [ ]:
merged = td.build_merged(data)print("merged shape:", merged.shape)merged.head(3)

In [ ]:
# 4.I - the most-loved spotspopularity = td.place_popularity(merged, min_ratings=cfg.top_k // 2)display(popularity.head(15)[    ["place_name", "city", "category", "n_ratings", "mean_rating", "bayesian_score"]])

**Why `bayesian_score` and not just the mean.** A place with two 5-star ratingsis not "the best destination in Indonesia" — it is a place almost nobody hasrated. The Bayesian score shrinks each place's average toward the global mean inproportion to how few ratings it has, so a genuinely well-liked spot with 30ratings outranks a fluke with 2. Sorting by raw mean would fill the top of thistable with noise, which is exactly the mistake this section exists to avoid.

In [ ]:
top15 = popularity.head(15).iloc[::-1]fig, ax = plt.subplots(figsize=(10, 6))bars = ax.barh(top15["place_name"], top15["bayesian_score"], color=ACCENT)for y, (score, n) in enumerate(zip(top15["bayesian_score"], top15["n_ratings"])):    ax.text(score, y, f"  {score:.2f}  (n={n})", va="center", fontsize=9, color="#444")ax.set(xlabel="Bayesian-adjusted rating", title="15 most-loved tourist spots")ax.set_xlim(left=min(top15["bayesian_score"]) - 0.15)fig.tight_layout()

In [ ]:
# 4.I continued - which city has the most-loved spots?city_love = (    merged.groupby("city")    .agg(n_ratings=("place_ratings", "size"),         mean_rating=("place_ratings", "mean"),         n_places=("place_id", "nunique"))    .round(3))city_love["top10_spots"] = [    (popularity.head(50)["city"] == c).sum() for c in city_love.index]display(city_love.sort_values("mean_rating", ascending=False))fig = plot_bar(    city_love.sort_values("mean_rating").index,    city_love.sort_values("mean_rating")["mean_rating"].round(3),    title="Average user rating by city", ylabel="mean rating",)

In [ ]:
# 4.II - which category do users like most?cat_love = (    merged.groupby("category")    .agg(n_ratings=("place_ratings", "size"),         mean_rating=("place_ratings", "mean"),         n_places=("place_id", "nunique"))    .round(3)    .sort_values("mean_rating", ascending=False))display(cat_love)fig, ax = plt.subplots(figsize=(9, 5))order = cat_love.sort_values("mean_rating")ax.barh(order.index, order["mean_rating"], color=ACCENT)for y, (v, n) in enumerate(zip(order["mean_rating"], order["n_ratings"])):    ax.text(v, y, f"  {v:.3f}  (n={n:,})", va="center", fontsize=9, color="#444")ax.set(xlabel="mean user rating", title="Which category do tourists like most?")ax.set_xlim(left=order["mean_rating"].min() - 0.1)fig.tight_layout()

**Read this one carefully.** The spread between the best and worst category issmall — a few hundredths of a rating point on a 1–5 scale. Before declaring awinner, check whether that gap is larger than the noise. The bootstrap belowdoes exactly that, and it is the difference between a defensible finding and aconfident-sounding artefact of rounding.

In [ ]:
# Is the category gap real? 95% bootstrap CI on each category's mean.rng = np.random.default_rng(cfg.seed)rows = []for category, group in merged.groupby("category"):    values = group["place_ratings"].to_numpy()    boots = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(2000)]    lo, hi = np.percentile(boots, [2.5, 97.5])    rows.append({"category": category, "mean": values.mean().round(3),                 "ci_low": round(lo, 3), "ci_high": round(hi, 3), "n": len(values)})ci = pd.DataFrame(rows).sort_values("mean", ascending=False).reset_index(drop=True)display(ci)fig, ax = plt.subplots(figsize=(9, 4.6))y = np.arange(len(ci))ax.hlines(y, ci["ci_low"], ci["ci_high"], color=ACCENT, lw=3, alpha=0.55)ax.scatter(ci["mean"], y, color=CONTRAST, zorder=5)ax.set_yticks(y, ci["category"])ax.invert_yaxis()ax.set(xlabel="mean rating (95% bootstrap CI)",       title="Category preference — do the intervals actually separate?")fig.tight_layout()

If those intervals overlap, the correct conclusion is *"users rate allcategories about the same"* — which is itself a useful finding for the agency:it means category is a weak marketing lever in this data, and personalisation(§7 onward) is where the value is.

## 6. Before modelling: does this data contain preference signal?A recommender can only learn structure that exists. Before building one — andcertainly before reporting that it "works" — establish whether these ratingscarry any preference information at all.Five independent checks, each of which a genuine ratings corpus should pass:1. **Shape** — human ratings are J-shaped: mostly 4s and 5s, mean around   4.0–4.3, negative skew. Near-uniform 1–5 with mean ≈ 3.0 is what a random   number generator produces.2. **Place effects** — if some places really are better, an ANOVA across places   should be significant.3. **Split-half reliability** — split the ratings randomly in two; a place's   mean in one half should predict its mean in the other. Measured against a   shuffled null, this is the most direct test available.4. **External validity** — `tourism_with_id` carries an independently sourced   `Rating` column. Real user ratings should correlate with it.5. **Random floor** — any recommender must beat picking places at random.This is not padding. If the checks fail, every number downstream has to be readdifferently — and knowing that *before* modelling is what stops you reportingnoise as a result.

In [ ]:
from src.evaluation import signaldiagnostics, verdict = signal.diagnose(ratings, merged, verbose=True)

In [ ]:
# Two legible views of the same conclusion.fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))# (a) observed rating distribution vs what humans actually produceobserved = ratings["place_ratings"].value_counts(normalize=True).sort_index()axes[0].bar(observed.index - 0.19, observed.values, width=0.38,            color=ACCENT, label="this dataset")# Typical J-shaped profile of a real 1-5 ratings corpus, shown for reference.human = pd.Series([0.05, 0.08, 0.17, 0.33, 0.37], index=[1, 2, 3, 4, 5])axes[0].bar(human.index + 0.19, human.values, width=0.38,            color=CONTRAST, alpha=0.75, label="typical human corpus")axes[0].set(xlabel="rating", ylabel="share of ratings",            title="Rating distribution: observed vs human-typical")axes[0].set_xticks([1, 2, 3, 4, 5])axes[0].legend()# (b) split-half reliability of place means against the shuffled nullsh = signal.split_half_reliability(ratings, n_shuffles=50)axes[1].axvspan(sh["shuffled_r_mean"] - 2 * sh["shuffled_r_sd"],                sh["shuffled_r_mean"] + 2 * sh["shuffled_r_sd"],                color="grey", alpha=0.20, label="noise band (shuffled ±2 SD)")axes[1].axvline(sh["shuffled_r_mean"], color="#888", ls="--", lw=1.5,                label=f"shuffled r = {sh['shuffled_r_mean']:.3f}")axes[1].axvline(sh["observed_r"], color=CONTRAST, lw=2.5,                label=f"observed r = {sh['observed_r']:.3f}")axes[1].set(xlabel="split-half correlation of place mean ratings", yticks=[],            title="Is a place's average rating reproducible?")axes[1].legend(loc="upper right", fontsize=9)fig.tight_layout()

### What this means for the rest of the notebookIf the verdict above is **NO USABLE PREFERENCE SIGNAL**, then:- The EDA rankings in §5 — most-loved spots, best city, most-liked category —  are orderings of **noise**. The bootstrap intervals already hinted at it;  these checks confirm it. Report them as *not statistically distinguishable*,  not as findings.- **No collaborative filtering model can beat a random recommender**, because  there is nothing to learn. That is a property of the data, not a failure of  the model or of the modelling.- The right response is not to tune hyperparameters until a number moves. It is  to build the model the brief asks for, evaluate it honestly against the random  floor, and explain why the result looks the way it does.That is a stronger submission than a fabricated win, and it is what an actualconsulting engagement would deliver. The models below are built exactly asspecified — we now simply know how to read their scores.

## 7. Model 1 — Item-based collaborative filtering — *brief task 5*This is the model that answers the brief's exact wording: *"recommend otherplaces to visit using the current tourist location (place name)."*How it works:1. Build the users × places rating matrix.2. **Mean-centre each user's row.** Without this, a generous user who rates   everything 5 and a strict user who rates everything 3 look like opposites,   when they may agree perfectly on rank order.3. Cosine similarity between place *columns*.4. Shrink similarities computed from few co-raters — two people who both rated   two places can produce a similarity of 1.0 that means nothing.

In [ ]:
train_r, test_r = td.train_test_split_ratings(ratings, cfg)print(f"train ratings: {len(train_r):,}   test ratings: {len(test_r):,}")print(f"users in train: {train_r['user_id'].nunique()}   places: {train_r['place_id'].nunique()}")

In [ ]:
item_cf = ItemBasedCF(min_ratings_per_place=cfg.min_ratings_per_place).fit(train_r, places)print("similarity matrix:", item_cf.similarity_.shape)

In [ ]:
# THE BRIEF'S CORE REQUIREMENT: given a place name, recommend other places.QUERY = places["place_name"].iloc[0]print(f"Because you visited: {QUERY}\n")item_cf.recommend_similar(QUERY, n=10)

In [ ]:
# A few more, including a partial-name lookup.for query in popularity.head(3)["place_name"]:    print("=" * 70)    print(f"Because you visited: {query}")    display(item_cf.recommend_similar(query, n=5))

## 8. Model 2 — Keras embedding matrix factorisationThe brief mandates TensorFlow for Part 1 and leaves Part 2 open; building thesecond recommender in Keras keeps the whole capstone on one stack.$$\hat{r}_{ui} = \sigma\big(\mathbf{p}_u \cdot \mathbf{q}_i + b_u + b_i\big)$$**On model size.** 300 users × 437 places with ~10k ratings is a *tiny*, ~92%sparse problem. A large neural recommender memorises it within a few epochs.Embedding dimension **32** with L2 regularisation and early stopping is thehonest configuration — and if the training curve still separates sharply fromvalidation, that is a finding to report, not a bug to hide.

In [ ]:
import tensorflow as tffrom src.models.recommender_net import (    KerasRecommender, RatingEncoder, build_recommender_net, train_recommender,)encoder = RatingEncoder().fit(train_r)x_train, y_train = encoder.transform(train_r)x_test, y_test = encoder.transform(test_r)print(f"users={encoder.n_users}  places={encoder.n_places}")print(f"train={x_train.shape}  test={x_test.shape}")mf = build_recommender_net(encoder.n_users, encoder.n_places, cfg)mf.summary()

In [ ]:
history = train_recommender(mf, x_train, y_train, x_test, y_test, cfg, verbose=1)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))ax.plot(history.history["rmse"], color=ACCENT, lw=2, label="train")ax.plot(history.history["val_rmse"], color=CONTRAST, lw=2, label="validation")best = int(np.argmin(history.history["val_rmse"]))ax.scatter([best], [history.history["val_rmse"][best]], color=CONTRAST, zorder=5)ax.annotate(f"best {history.history['val_rmse'][best]:.4f} (ep {best + 1})",            (best, history.history["val_rmse"][best]),            textcoords="offset points", xytext=(8, 6), fontsize=9, color=CONTRAST)ax.set(xlabel="epoch", ylabel="RMSE (scaled 0-1)", title="RecommenderNet training")ax.legend()fig.tight_layout()

In [ ]:
keras_rec = KerasRecommender(mf, encoder, places)sample_user = int(train_r["user_id"].value_counts().index[0])seen = set(train_r[train_r["user_id"] == sample_user]["place_id"])print(f"User {sample_user} has rated {len(seen)} places. Top-10 recommendations:\n")keras_rec.recommend_for_user(sample_user, n=10, exclude_seen=seen)

In [ ]:
# The neural model answers the place-to-place question too, via its learned# embeddings - a useful cross-check on the item-CF result in §7.print(f"Places similar to '{QUERY}' — learned embedding space\n")keras_rec.similar_places(QUERY, n=10)

### Do the two models agree?If item-CF and the neural embeddings independently surface the same neighbours,that is genuine signal. If they disagree completely on a ~92% sparse matrix,both are largely fitting noise — and the report should say so.

In [ ]:
def overlap_at_k(place_name, k=10):    a = set(item_cf.recommend_similar(place_name, n=k)["place_id"])    b = set(keras_rec.similar_places(place_name, n=k)["place_id"])    return len(a & b) / ksample_places = popularity.head(20)["place_name"].tolist()scores = []for name in sample_places:    try:        scores.append(overlap_at_k(name, k=10))    except (KeyError, ValueError):        continueif scores:    print(f"Mean top-10 overlap between the two models: {np.mean(scores):.1%}")    print(f"(across {len(scores)} popular places; 10% would be chance-level)")else:    print("No place could be scored by both models - check the cold-start filters.")

## 9. Evaluation — do the models beat a dumb baseline?

In [ ]:
# The bar every recommender must clear: recommend the globally most popular# places to everyone. On small datasets this is a surprisingly strong strategy.baseline = PopularityBaseline(min_ratings=5).fit(train_r)rating_results = {    "Popularity baseline": R.rating_metrics(        test_r["place_ratings"], baseline.predict(test_r["user_id"], test_r["place_id"])),    "Item-based CF": R.rating_metrics(        test_r["place_ratings"], item_cf.predict(test_r["user_id"], test_r["place_id"])),    "Keras MF": R.rating_metrics(        test_r["place_ratings"], keras_rec.predict(test_r["user_id"], test_r["place_id"])),}display(R.compare_models(rating_results))

In [ ]:
# Ranking quality - what the tourist actually sees.n_places_total = places["place_id"].nunique()ranking_results = {}for label, fn in [    ("Random floor", signal.make_random_recommend_fn(places["place_id"].to_numpy())),    ("Popularity baseline", R.make_popularity_recommend_fn(baseline)),    ("Item-based CF", R.make_itemcf_recommend_fn(item_cf)),    ("Keras MF", R.make_keras_recommend_fn(keras_rec)),]:    print(f"\n--- {label} ---")    ranking_results[label] = R.evaluate_topk(        fn, train_r, test_r, k=cfg.top_k,        relevance_threshold=cfg.relevance_threshold,        n_places_total=n_places_total, verbose=True,    )comparison = R.compare_models(ranking_results)display(comparison)

In [ ]:
metrics_to_plot = [f"precision@{cfg.top_k}", f"recall@{cfg.top_k}", f"ndcg@{cfg.top_k}"]fig, ax = plt.subplots(figsize=(10, 5))width, x = 0.25, np.arange(len(metrics_to_plot))for i, (label, colour) in enumerate(zip(comparison.index, PALETTE)):    values = [comparison.loc[label, m] for m in metrics_to_plot]    bars = ax.bar(x + i * width, values, width, label=label, color=colour)    for b, v in zip(bars, values):        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}",                ha="center", va="bottom", fontsize=8, color="#444")ax.set_xticks(x + width, metrics_to_plot)ax.set(ylabel="score", title=f"Top-{cfg.top_k} ranking quality vs random floor and popularity baseline")ax.legend()fig.tight_layout()

**How to interpret this honestly.**- If a model beats the popularity baseline on **NDCG@10**, personalisation is  adding real value and the agency should deploy it.- If it does not, the correct conclusion is that with ~10k ratings there is not  enough signal to personalise, and simply promoting popular destinations is  the better strategy *until more interaction data is collected*. That is a  legitimate, defensible result — not a failure of the assignment.- **Catalogue coverage** is the check on that verdict. The popularity baseline  recommends the same handful of places to everyone, so its coverage is near  zero. A recommender with similar accuracy but far higher coverage is more  valuable to the agency, because it spreads visitors across the country  instead of funnelling everyone to the same five sites.## 10. The deliverable — a recommendation functionWhat the agency actually asked for: a tourist is standing somewhere; tell themwhere to go next.

In [ ]:
def recommend_next_destination(place_name: str, n: int = 5, user_id: int | None = None):    # Recommend places to visit next, given where the tourist is now.    #    # Uses item-based CF for the place-to-place signal (the brief's requirement).    # If a user_id is supplied, the neural model's personalised list is shown    # alongside it for comparison.    print("=" * 72)    print(f"CURRENT LOCATION: {place_name}")    print("=" * 72)    try:        info = places[places["place_name"].str.lower() == place_name.lower()]        if len(info):            row = info.iloc[0]            print(f"  {row['category']} · {row['city']} · listed rating {row['rating']}")    except Exception:        pass    print(f"\nTourists who enjoyed this also enjoyed:\n")    recs = item_cf.recommend_similar(place_name, n=n)    display(recs)    if user_id is not None and user_id in encoder.user_to_idx:        seen = set(train_r[train_r["user_id"] == user_id]["place_id"])        print(f"\nPersonalised for user {user_id} (neural model):\n")        display(keras_rec.recommend_for_user(user_id, n=n, exclude_seen=seen))    return recs_ = recommend_next_destination(QUERY, n=5, user_id=sample_user)

In [ ]:
# Save artefacts.from src.config import ARTIFACT_ROOTimport jsonARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)mf.save(ARTIFACT_ROOT / "recommender_net.keras")item_cf.similarity_.to_csv(ARTIFACT_ROOT / "item_similarity.csv")summary = {    "n_users": int(users["user_id"].nunique()),    "n_places": int(places["place_id"].nunique()),    "n_ratings": int(len(ratings)),    "sparsity": round(1 - len(ratings) / (users["user_id"].nunique() * places["place_id"].nunique()), 4),    "rating_metrics": rating_results,    "ranking_metrics": ranking_results,    "cleaning_log": data["_log"],}(ARTIFACT_ROOT / "part2_results.json").write_text(json.dumps(summary, indent=2, default=str))print(json.dumps({k: summary[k] for k in ["n_users", "n_places", "n_ratings", "sparsity"]}, indent=2))

## 11. Conclusions*Fill the bracketed figures in from your run before submitting.***EDA findings (tasks 2–4).**1. The rating population is young and Java-centric — campaigns built on this   data speak to domestic visitors in their twenties and thirties, and should   not be extrapolated to other segments.2. `[CITY]` offers the most nature attractions by volume, while `[CITY2]` has   the highest concentration of them. Which is "best for a nature enthusiast"   depends on whether the visitor wants choice or focus.3. Category-level rating differences are **small**, and the bootstrap intervals   in §5 show whether they separate at all. If they overlap, category is a weak   marketing lever and personalisation is where the value lies.4. The most-loved spots are ranked by a Bayesian-adjusted score, so   low-volume flukes do not crowd out genuinely popular destinations.**Recommender findings (task 5).**| Model | RMSE | NDCG@10 | Coverage ||---|---|---|---|| Popularity baseline | `[…]` | `[…]` | `[…]` || Item-based CF | `[…]` | `[…]` | `[…]` || Keras MF | `[…]` | `[…]` | `[…]` |The two collaborative models agreed on **`[X]%`** of their top-10 neighbours,which indicates `[genuine shared signal / mostly noise]`.**Limitations worth stating in the report.**- **Sparsity.** ~92% of the user–place matrix is empty. Every conclusion here  is drawn from roughly 10,000 interactions and should be revisited as the  agency collects more.- **Cold start.** Neither model can recommend a place nobody has rated, nor  serve a brand-new user. A content-based model over `Category`/`City`/  `Description` would cover that gap — the natural next extension.- **No timestamps.** The train/test split is random rather than chronological,  so it cannot detect whether tastes drift over time.- **Popularity bias.** Ratings concentrate on already-famous places, so the  system will tend to reinforce existing tourist flows. If the agency's goal is  to spread visitors to lesser-known heritage sites, coverage and novelty need  to be optimised explicitly — not just accuracy.